In [1]:
import nbformat

# Path to your notebook
notebook_path = "interpret.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

for i, cell in enumerate(nb.cells):
    # print("="*80)
    print(f"Cell {i}")
    # print(f"Type       : {cell.cell_type}")
    # print(f"Execution# : {cell.get('execution_count', None)}")
    # print(f"Metadata   : {cell.get('metadata', {})}")

    # Print the source (truncate if very long)
    src = cell.get("source", "")
    if src.strip() == "":
        pass
        # print("[Empty cell]")
    else:
        # print("--- Source ---")
        print(src)

    # If it's a code cell, also print outputs if any
    if cell.cell_type == "code":
        # print("--- Outputs ---")
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                print("[stream]", out.text.strip())

            # Check both output types that contain rich data
            elif out.output_type in ["execute_result", "display_data"]:
                data = out.get("data", {})
                
                # Prioritize printing markdown if it exists
                if "text/markdown" in data:
                    print("[markdown_output]\n", data["text/markdown"])
                
                # Fallback to plain text if no markdown is found
                elif "text/plain" in data:
                    print("[text_output]", data["text/plain"])

            elif out.output_type == "error":
                print("[error]", "".join(out.get("traceback", [])))
    print()


Cell 0
import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import os
from IPython.display import display, HTML
import matplotlib.patheffects as PathEffects

# --- CONFIGURATION ---
DB_PATH = "db/results.sqlite" 
sns.set_theme(style="whitegrid", context="paper")

Cell 1
# --- CUSTOM CSS FOR DASHBOARD ---
DASHBOARD_CSS = """
<style>
    :root {
        --bg-color: #ffffff;
        --text-color: #2c3e50;
        --accent: #3498db;
        --border: #e0e0e0;
        --success: #27ae60;
        --warning: #f39c12;
        --danger: #c0392b;
    }
    .db-container { font-family: 'Segoe UI', sans-serif; max-width: 100%; color: var(--text-color); }
    .db-header { background: #f8f9fa; padding: 15px; border-bottom: 2px solid var(--accent); margin-bottom: 20px; }
    .db-stat-grid { display: grid; grid-template-columns: re